In [1]:
import os
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import nibabel as nib

data_root = "data/lidc_process/"
case = "LIDC-IDRI-1001"
image = nib.load(os.path.join(data_root, case, case + "_volume.nii.gz")).get_fdata().transpose(2, 1, 0)
nodule_mask = nib.load(os.path.join(data_root, case, case + "_nodule_mask.nii.gz")).get_fdata().transpose(2, 1, 0)

print("sample_id:", case)
print("slice file:", case + "_volume.nii.gz", "| shape:", image.shape)
print("mask file :", case + "_nodule_mask.nii.gz",  "| shape:", nodule_mask.shape)
print("slices with nodules:", np.unique(np.argwhere(nodule_mask)[:, 0]))

for idx in range(image.shape[0]):
    img = image[idx]

    fig, axes = plt.subplots(1, 2, figsize=(10, 5))
    axes[0].imshow(img, cmap="gray")
    axes[0].set_title(f"Slice {idx}")
    axes[0].axis("off")

    axes[1].imshow(img, cmap="gray")
    axes[1].set_title(f"Slice {idx} + Mask")
    axes[1].axis("off")

    # Case 1: pixel-wise mask volume
    if nodule_mask.ndim >= 3 and nodule_mask.shape[0] == image.shape[0] and nodule_mask[idx].shape == img.shape:
        axes[1].imshow(nodule_mask[idx] > 0, cmap="Reds", alpha=0.35)

    # Case 2: per-slice center/box annotations: [x, y, radius_x, radius_y]
    elif nodule_mask.ndim >= 2 and nodule_mask.shape[0] == image.shape[0]:
        centers = nodule_mask[idx]
        for center in centers:
            if len(center) < 4:
                continue
            x, y, radius_x, radius_y = center[:4]
            if radius_x <= 0 or radius_y <= 0:
                continue
            rect = patches.Rectangle(
                (x - radius_x, y - radius_y),
                2 * radius_x,
                2 * radius_y,
                linewidth=2,
                edgecolor="red",
                facecolor="none"
            )
            axes[1].add_patch(rect)

    plt.tight_layout()
    plt.show()

FileNotFoundError: No such file or no access: 'data/lidc_process/LIDC-IDRI-1001/LIDC-IDRI-1001_volume.nii.gz'